# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Task type: Ranking / Scoring

My FlyRank lane is content-refresh prioritization.

The decision I want to improve is: **Which content items should an editor review first?**

I would build a ranking/scoring system that gives each content item a priority score representing how likely it is to decline in a future evaluation window.

This is a ranking problem because the main operational question is not simply "Will this page decline, yes or no?" The more useful question is "Which pages should we put at the top of the editor's review queue?"

The ranking would allow an editor to focus limited time on the highest-priority content items first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/13aakash/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 42), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.87 MiB | 6.43 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [ ]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [ ]:
!ls data/raw

content_refresh_anonymized.csv


In [ ]:
import pandas as pd

# Load the FlyRank starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Shape: (30000, 44)
Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
# First, inspect the actual columns in the dataset

print("Dataset shape:", df.shape)
print("\nAll columns:")
print(df.columns.tolist())

print("\nColumns related to decline/trend:")
print([
    col for col in df.columns
    if "declin" in col.lower() or "trend" in col.lower()
])

Dataset shape: (30000, 44)

All columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Columns related to decline/trend:
['trend_direction', 'trend_pct']


The pipeline's `is_declining_label` (computed downstream from `trend_direction` in `01_prepare_features.py` — not present in this raw CSV) demonstrates the binary shape of the target I would eventually predict: one observed outcome per content item.

For the final predictive setup, the label should be aligned to a future outcome window so that the features represent information available before the outcome occurs.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


### Target: Future content decline

The outcome I ultimately want to predict is whether a content item will decline in a future evaluation window.

The intended target would be a binary outcome:

- `1` = the content item is observed to decline in the future evaluation window
- `0` = the content item is not observed to decline in the future evaluation window

The important point is that the target should come from an observed future outcome, not from a manually written priority rule.

The starter dataset does not contain a separate `is_declining_label` column. It contains `trend_direction` and `trend_pct`, but I will not use these as predictive features because they are directly related to the current trend/decline outcome.

For this framing exercise, I am therefore defining the future observed decline as the intended target and using the starter dataset to demonstrate the unit of analysis and the available content-level signals.

A real predictive setup would need a future outcome window so that the input features are measured before the outcome occurs.
### Target column sketch

The eventual modeling dataset would contain a target column like this:

| content_id | future_decline |
|---|---:|
| content_001 | 1 |
| content_002 | 0 |
| content_003 | 0 |
| content_004 | 1 |
| content_005 | 0 |

Here, `1` means the content item was later observed to decline and `0` means it was not.

These example values are only a sketch of the target structure. They are not labels taken from the starter dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary metric: Precision@K

I would use **Precision@K** as the primary success metric.

The operational situation is that editors have limited time, so they cannot review every content item. The model should therefore put the most useful items near the top of the queue.

For example, if `K = 100`, Precision@100 asks:

> Of the 100 content items ranked highest by the model, what proportion actually declined in the future evaluation window?

A higher Precision@100 would mean that the editor's limited review time is being spent on a more useful set of pages.

I would define the model as useful if its top-K ranking has meaningfully better precision than a simple baseline, rather than choosing a success threshold after seeing the results.

### Formula

**Precision@K = Number of true declining items among the top K ranked items / K**

For example, if 35 of the top 100 ranked pages actually decline:

**Precision@100 = 35 / 100 = 35%**

### Baseline

The ML ranking should eventually be compared against a simple fixed-rule baseline.

For example, a baseline could rank pages using one existing signal or a simple manually defined rule.

The ML approach is useful only if it identifies more future declines in the top-K queue than the baseline while using information available before the outcome window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


### Unit of analysis

**One row = one pseudonymized content item.**

The starter dataset contains one row per content item, with content-level performance and descriptive signals.

For this framing exercise, the content item is the thing that an editor would potentially prioritize for review.

The client ID identifies which client the content belongs to, but it is not itself a predictive feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could be useful as a baseline, but it becomes difficult to maintain when content decline depends on several signals at the same time.

For example, an editor could create a rule such as:

> "Prioritize a page if its CTR is below a certain value and its average position is above a certain value."

The problem is that the same signal may mean different things depending on the content type, client, engagement behavior, search performance, and other available signals.

The useful pattern may therefore involve combinations of several variables rather than one simple threshold.

ML could learn these multi-signal relationships from historical outcomes and convert them into a priority ranking.

The reason to use ML is not simply that it is more advanced. It earns its place if it can produce a top-K queue with better Precision@K than a reasonable fixed-rule baseline.

If a simple rule performs just as well, the rule would be preferable because it is easier to understand and maintain.

### Decision and action

The output would be a ranked list of content items, from highest to lowest priority.

An editor would use this ranking to decide which pages to inspect and potentially refresh first.

The model does not automatically rewrite or remove content. It supports the editor's decision by helping allocate limited review time toward pages with higher predicted risk.

### Cost of wrong predictions

A false positive means an editor spends time reviewing a page that may not actually decline.

A false negative means a page that later declines may be missed or reviewed too late.

For this use case, the ranking should therefore be judged by how effectively it concentrates genuinely declining pages near the top of the review queue.

## Final problem frame

For **content editors**, deciding **which content items to review first**, I would build a **ranking/scoring system** from historical content-performance data, scoring each item by its likelihood of **declining in a future evaluation window**, measured primarily by **Precision@K**. A wrong call can waste editor review time through false positives or allow a genuinely declining page to be missed through false negatives. A plain rule may not be enough because the useful pattern can involve multiple interacting signals such as search performance, engagement, content type, and other content-level characteristics. However, ML would only earn its place if it produces a meaningfully better top-K ranking than a reasonable fixed-rule baseline. The intended result is **decision support**, not automatic content changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.